In [ ]:
import os
import sys; sys.path.append(os.path.join(os.path.abspath(''), '../unit_cell_experiments/'));

import experiment_helper
import igl
from periodic_simulation_setup import *
import json

import parallelism, multiprocessing, itertools, setproctitle
import os, time, numpy as np

import json


In [ ]:
allowBending = False
useTFT = True
disableFusedRegionTFT = False
stiffness_pressure = 0.3
scale_factor_pressure = 0.01
avg_len = 0.15


In [ ]:
tag = '0.00_2.5_75'
amp = float(tag.split('_')[0])
r = float(tag.split('_')[1])
angle = float(tag.split('_')[2])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def equilateral_triangle_vertices(radius, center=(0, 0)):
    """Generate vertices of an equilateral triangle given a radius."""
    angles = np.linspace(0, 2 * np.pi, 4)[:-1]  # 0, 120, 240 degrees
    return np.array([center[0] + radius * np.cos(angles), center[1] + radius * np.sin(angles)]).T

def square_vertices(radius, center=(0, 0)):
    """Generate vertices of an equilateral triangle given a radius."""
    angles = np.linspace(0, 2 * np.pi, 5)[:-1]  # 0, 120, 240 degrees
    return np.array([center[0] + radius * np.cos(angles), center[1] + radius * np.sin(angles)]).T

num_sides = 4

if num_sides == 4:
    boundary = square_vertices
else:
    boundary = equilateral_triangle_vertices
    
def rotate_vertices(vertices, angle, center=(0, 0)):
    """Rotate vertices around a center by a given angle."""
    angle_rad = np.radians(angle)
    rotation_matrix = np.array([[np.cos(angle_rad), -np.sin(angle_rad)],
                                [np.sin(angle_rad), np.cos(angle_rad)]])
    return np.dot(vertices - center, rotation_matrix.T) + center

def draw_sine_curve(ax, start, end, amplitude, resolution):
    """Draw a full period of a sine curve from start to end with a given resolution and return vertices and edges."""
    # Calculate the distance and angle between start and end points
    distance = np.linalg.norm(np.array(end) - np.array(start))
    angle = np.arctan2(end[1] - start[1], end[0] - start[0])
    
    # Generate x-values between start and end points
    x_values = np.linspace(0, distance, resolution)
    
    # Generate y-values for a full period of sine curve with given amplitude
    y_values = amplitude * np.sin(2 * np.pi * x_values / distance)
    
    # Rotate and translate the sine curve to fit between start and end points
    sine_curve = np.array([x_values, y_values]).T
    rotation_matrix = np.array([[np.cos(angle), -np.sin(angle)],
                                [np.sin(angle), np.cos(angle)]])
    sine_curve_rotated = np.dot(sine_curve, rotation_matrix.T) + start
    
    # Plot the sine curve
    ax.plot(sine_curve_rotated[:, 0], sine_curve_rotated[:, 1], 'b-')
    
    # Prepare vertices and edges
    vertices = sine_curve_rotated.tolist()
    edges = [(i, i + 1) for i in range(len(vertices) - 1)]
    
    return vertices, edges

def main(r1, r2, beta, alpha, amp, res, period_offset):
    fig, ax = plt.subplots()
    ax.set_aspect('equal')

    # Inner triangle
    inner_vertices = boundary(r1)
    inner_vertices = rotate_vertices(inner_vertices, beta)

    ax.plot(*np.append(inner_vertices, [inner_vertices[0]], axis=0).T, 'r-')

    # Outer triangle
    outer_vertices = boundary(r2)
    outer_vertices_rotated = rotate_vertices(outer_vertices, alpha)
    ax.plot(*np.append(outer_vertices_rotated, [outer_vertices_rotated[0]], axis=0).T, 'g-')

    # Draw sine curves connecting each inner vertex to the corresponding outer vertex
    all_vertices = []
    all_edges = []
    for i in range(4):
        sine_vertices, sine_edges = draw_sine_curve(ax, inner_vertices[i], outer_vertices_rotated[(i + 1) % num_sides], amp, res)
        start_index = len(all_vertices)
        all_vertices.extend(sine_vertices)
        all_edges.extend([(start_index + edge[0], start_index + edge[1]) for edge in sine_edges])
        
        horizontal_shift = (inner_vertices[i] - outer_vertices_rotated[(i + 1) % num_sides]) * 1
        vertical_shift = rotate_vertices(inner_vertices[i] - outer_vertices_rotated[(i + 1) % num_sides], 90) * 0.2
        
        offset = 2
        # shifts = [vertical_shift + 0.05 * horizontal_shift]
        shifts = []
        distance = np.linalg.norm(inner_vertices[i]- outer_vertices_rotated[(i + 1) % num_sides]) 

        period = distance + period_offset
        for sign_i in [1, 0, -1]:
            for sign_j in [1, 0, -1]:
                if sign_i == 0 and sign_j == 0:
                    continue
                # shifts.append(np.array([offset * sign_i, offset * sign_j]))
                # shifts.append(np.array([offset * sign_i, offset * sign_j]) + 0.05 * horizontal_shift + vertical_shift)

                
                  # horizontal_shift - vertical_shift,
                  # horizontal_shift, 
                  # vertical_shift * 3 + 0.5 * horizontal_shift,
                  # vertical_shift * 2 + 0.5 * horizontal_shift]
                shifts.append(np.array([period * sign_i, period * sign_j]))

        # shifts = [np.array([distance + 0.5, 0]), np.array([0, distance + 0.5])]
        for shift in shifts:
            start_index = len(all_vertices)
            new_curve = sine_vertices + shift
            all_vertices.extend(new_curve)
            all_edges.extend([(start_index + edge[0], start_index + edge[1]) for edge in sine_edges])
            ax.plot(new_curve[:, 0], new_curve[:, 1], 'b-')


    plt.show()
    return all_vertices, all_edges, period

# Parameters
r1 = 1
r2 = r1 * 2.5
beta = -60
alpha = -55  # degrees
amp = 0.3  # amplitude of the sine curve
res = 20  # resolution
period_offset = 0.7

vertices, edges, period = main(r1, r2, beta, alpha, amp, res, period_offset)

In [ ]:
import ipywidgets as widgets
from IPython.display import display

In [ ]:

# Create sliders for the variables
r1_slider = widgets.FloatSlider(value=1, min=1, max=3, step=0.1, description='r1')
r2_slider = widgets.FloatSlider(value=2.0, min=1, max=3, step=0.1, description='r2')
beta_slider = widgets.FloatSlider(value=45, min=0, max=90, step=1, description='beta')
alpha_slider = widgets.FloatSlider(value=45, min=0, max=90, step=1, description='alpha')
amp_slider = widgets.FloatSlider(value=0.2, min=0, max=1, step=0.01, description='amp')
res_slider = widgets.IntSlider(value=15, min=10, max=500, step=10, description='res', disabled=True)
period_offset_slider = widgets.FloatSlider(value=0.5, min=0, max=2, step=0.1, description='period_offset')

# Create an interactive widget
interactive_plot = widgets.interactive(main, r1=r1_slider, r2=r2_slider, beta=beta_slider, alpha=alpha_slider, amp=amp_slider, res=res_slider, period_offset=period_offset_slider)

# Display the interactive widget
display(interactive_plot)

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt

# def equilateral_triangle_vertices(radius, center=(0, 0)):
#     """Generate vertices of an equilateral triangle given a radius."""
#     angles = np.linspace(0, 2 * np.pi, 4)[:-1]  # 0, 120, 240 degrees
#     return np.array([center[0] + radius * np.cos(angles), center[1] + radius * np.sin(angles)]).T

# def square_vertices(radius, center=(0, 0)):
#     """Generate vertices of an equilateral triangle given a radius."""
#     angles = np.linspace(0, 2 * np.pi, 5)[:-1]  # 0, 120, 240 degrees
#     return np.array([center[0] + radius * np.cos(angles), center[1] + radius * np.sin(angles)]).T

# num_sides = 4

# def rotate_vertices(vertices, angle, center=(0, 0)):
#     """Rotate vertices around a center by a given angle."""
#     angle_rad = np.radians(angle)
#     rotation_matrix = np.array([[np.cos(angle_rad), -np.sin(angle_rad)],
#                                 [np.sin(angle_rad), np.cos(angle_rad)]])
#     return np.dot(vertices - center, rotation_matrix.T) + center

# def draw_semi_circle(ax, start, end, resolution):
#     """Draw a semi-circle from start to end with a given resolution and return vertices and edges."""
#     mid_point = (start + end) / 2
#     radius = np.linalg.norm(start - mid_point)
#     angle = np.arctan2(end[1] - start[1], end[0] - start[0])
#     angles = np.linspace(angle, angle + np.pi, resolution)
#     semi_circle = np.array([mid_point[0] + radius * np.cos(angles), mid_point[1] + radius * np.sin(angles)]).T
#     new_angles = np.linspace(angles[0] + np.pi + np.pi / 6, angles[0] + np.pi, int(resolution / 2))
#     new_radius = radius * 2
#     new_center = semi_circle[0] - (mid_point - semi_circle[0]) / radius * new_radius
#     next_curve = np.array([new_center[0] + new_radius * np.cos(new_angles), new_center[1] + new_radius * np.sin(new_angles)]).T
#     curve = np.concatenate([next_curve[:-1], semi_circle])

#     ax.plot(curve[:, 0], curve[:, 1], 'b-')
#     return curve

# def main(r1, r2, alpha, res):
#     fig, ax = plt.subplots()
#     ax.set_aspect('equal')

#     # Inner triangle
#     inner_vertices = square_vertices(r1)
#     ax.plot(*np.append(inner_vertices, [inner_vertices[0]], axis=0).T, 'r-')

#     # Outer triangle
#     outer_vertices = square_vertices(r2)
#     outer_vertices_rotated = rotate_vertices(outer_vertices, alpha)
#     ax.plot(*np.append(outer_vertices_rotated, [outer_vertices_rotated[0]], axis=0).T, 'g-')

#     # Draw semi-circles connecting each inner vertex to the next outer vertex
#     # all_vertices = inner_vertices.tolist() + outer_vertices_rotated.tolist()
#     all_vertices = []
#     all_edges = []
#     vertex_index = len(all_vertices)
#     for i in range(num_sides):
#         semi_circle = draw_semi_circle(ax, inner_vertices[i], outer_vertices_rotated[(i + 1) % num_sides], res)
#         semi_circle_vertices = semi_circle.tolist()
#         all_vertices.extend(semi_circle_vertices)
#         start_index = vertex_index
#         for j in range(len(semi_circle_vertices) - 1):
#             all_edges.append([start_index + j, start_index + j + 1])
#         vertex_index += len(semi_circle_vertices)

#     plt.show()
#     return all_vertices, all_edges

# # Parameters
# r1 = 1
# r2 = 2.2
# alpha = 90  # degrees
# res = 30  # resolution

# vertices, edges = main(r1, r2, alpha, res)

In [ ]:
def run_main_with_current_values():
    current_values = interactive_plot.kwargs
    return main(**current_values)

# Run the main function with current slider values
vertices, edges, period = run_main_with_current_values()

In [ ]:
import visualization

In [ ]:
visualization.plot_line_segments(vertices, edges)

In [ ]:
vertices = np.concatenate([vertices, np.zeros((len(vertices), 1))], axis = 1)

In [ ]:
import mesher_helper

In [ ]:
w = period * 2
h = period * 2

In [ ]:
period

In [ ]:
boundary_vxs = np.array([[-w/2., -h/2., 0.], [w/2., -h/2., 0.], [w/2., h/2., 0.], [-w/2., h/2., 0.]])
mid_point = np.array([0, 0])
boundary_lines = np.array([[0, 1], [1, 2], [2, 3], [3, 0]]) + 1

In [ ]:
boundary_vxs

In [ ]:
import mesher_helper, importlib
importlib.reload(mesher_helper)

In [ ]:
v, f, fusing_data = mesher_helper.generate_mesh_from_embeddings_array_input_allow_boundary(0.1, 0.1, None, boundary_vxs, boundary_lines, np.array(vertices), np.array(edges) + 1)

In [ ]:
len(v)

In [ ]:
edges_from_faces = []
for face in (f):
    edges_from_faces.append([face[0], face[1]])
    edges_from_faces.append([face[1], face[2]])
    edges_from_faces.append([face[2], face[0]])

In [ ]:
len(f)

In [ ]:
m = MeshFEM.Mesh(v, np.array(f) - 1)

In [ ]:

marker = fusing_data 

finalMarkers = np.where(np.array(marker) == 1)[0]
# m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, flip_orientation= -1)
# m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1, flip_orientation= -1)

# vertices = m.vertices()
# vertices *= 4
# m = MeshFEM.mesh.Mesh(vertices, m.elements())
fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)


In [ ]:
visualization.plot_2d_mesh(m, pointList=fusedVtx)

In [ ]:
%%capture

ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = fusedVtx, epsilon = 1e-9)

In [ ]:
max(m.edgeLengths()), min(m.edgeLengths())

In [ ]:
# m = MeshFEM.Mesh('../experiments/parallelized_experiments/output/mirror_cosine_dash/2024_01_07_17_10/0.00_1.70_87.00/mesh_mirror_cosine_dash_0.00_1.70_87.00.obj')

In [ ]:
# dofs = np.load('../experiments/parallelized_experiments/output/mirror_cosine_dash/2024_01_07_15_35/0.00_0.70_82.00/mirror_cosine_dash_dofs_after_negative_stiffness_escape_0.00_0.70_82.00.npy')

In [ ]:
# visualization.plot_2d_mesh(m, pointList=finalMarkers, width=10, height=10)

In [ ]:
viewer = TriMeshViewer(ipu, width=1024, height=1024)
viewer.showWireframe(True)
viewer.show()

In [ ]:
configure_solver_parallelism()

In [ ]:
viewer.update()
viewer.showWireframe(True)

In [ ]:
framerate = 5 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])

In [ ]:
hessianShiftForRigidMotion = 1e-10
hessianShiftForAlphainPlanar = 1e-12

In [ ]:

def helper_run_equilibrium(ipu, allow_bending,  stiffness_pressure, cb, disableFusedRegionTFT = False, useTFT = True):
    # Choose strategy for constraining rigid motion
    # We first use no fix vars to check whether the equilibrium converge to a planar state, use a hessian shift rather than applying fixed vars to constrain the rigid motion so that the equilibrium solve converges faster. 
    bending_fixed_vars = [] if allow_bending else [ipu.numVars() - 2]
    fixedVars, hessianShift = [ipu.numVars() - 2], hessianShiftForRigidMotion

    ipu.sheet.setUseTensionFieldEnergy(useTFT)
    ipu.sheet.setUseHessianProjectedEnergy(False)
    if (disableFusedRegionTFT):
        ipu.sheet.disableFusedRegionTensionFieldTheory(False)
    ipu.sheet.pressure = stiffness_pressure

    opts.niter = 100
    opts.gradTol = 1e-10
    cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb, hessianShift = hessianShift)

    opts.niter = 100
    # Solve for true equilibrium with a much smaller hessian shift that's only used for the alpha variable when kappa becomes zero. Pin down the the x, y, z value of a center vertex. 
    fixedVars, hessianShift = list(periodic_unit_helper.get_center_fixedVars(ipu)) + [ipu.numVars() - 2], hessianShiftForAlphainPlanar
    cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb, hessianShift = hessianShift)

    opts.niter = 1000
    # Solve for true equilibrium with a much smaller hessian shift that's only used for the alpha variable when kappa becomes zero. Pin down the the x, y, z value of a center vertex. 
    fixedVars, hessianShift = list(periodic_unit_helper.get_center_fixedVars(ipu)) + bending_fixed_vars, hessianShiftForAlphainPlanar
    cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb, hessianShift = hessianShift)
    return cr

In [ ]:
helper_run_equilibrium(ipu, True, 0.3, cb)

In [ ]:
import periodic_simulation_setup

In [ ]:
periodic_simulation_setup.get_deformation_scale_factors(ipu)

In [ ]:
2 / np.pi